***ATHLETES ARCHETYPE PROJECT - Khang***
In this project I will examine different possible archetypes that many triathletes developed through their results to understand race strategies in triathlon as well as how each strategies benefits a race
We will examine the race data from IRONMAN 70.3 world championship in 2026

In [1]:
import pandas as pd

df = pd.read_csv("nice_ironman_results_2026.csv")

print(df.shape)
df.head()

(3498, 33)


,Bib Number,Athlete Name,Gender,City,State,Country,Division,Status,Finish Time,Swim Time,...,Bike Rank (Division),Run Rank (Overall),Run Rank (Gender),Run Rank (Division),Finish (Seconds),Swim (Seconds),T1 (Seconds),Bike (Seconds),T2 (Seconds),Run (Seconds)
0,44,Hayden Wilde,Male,Coastlands,North Island,NZ,MPRO,FIN,3:50:51,0:23:44,...,2,1,1,1,13851.0,1424.0,135.0,8210.0,41.0,4041.0
1,2,Kristian Blummenfelt,Male,bergen,NaN,NO,MPRO,FIN,3:51:09,0:23:50,...,16,2,2,2,13869.0,1430.0,135.0,8342.0,51.0,3912.0
2,1,Jelle Geens,Male,Zolder,BE-VLI,BE,MPRO,FIN,3:52:40,0:23:39,...,12,3,3,3,13960.0,1419.0,123.0,8331.0,55.0,4032.0
3,6,Gustav Iden,Male,Tertnes,NaN,NO,MPRO,FIN,3:53:16,0:24:40,...,6,4,4,4,13996.0,1480.0,128.0,8203.0,53.0,4131.0
4,43,Lasse Nygaard Priester,Male,Freiburg im Breisgau,BW,DE,MPRO,FIN,3:54:08,0:23:41,...,4,5,5,5,14048.0,1421.0,120.0,8233.0,51.0,4222.0


In [2]:
df["Gender"].value_counts()

Gender
Male      3450
Female      38
Name: count, dtype: int64

In [3]:
missing_finish = df[df["Finish Time"].isna()]

missing_finish["Status"].value_counts()
missing_finish["Division"].value_counts().head(20)

Division
M40-44    42
M45-49    40
M55-59    39
M35-39    38
M50-54    33
M60-64    31
M65-69    21
M70-74    21
M30-34    20
M18-24    20
M25-29    16
MPRO       5
M75-79     4
M80-84     2
MGuide     2
PC/ID      2
Name: count, dtype: int64

In [4]:
missing_finish[
    ["Athlete Name", "Gender", "Division", "Status",
     "Swim Time", "Bike Time", "Run Time", "Finish Time"]
].head(20)

,Athlete Name,Gender,Division,Status,Swim Time,Bike Time,Run Time,Finish Time
3146,Mathieu Gallet,Male,M25-29,DNF,0:27:33,NaN,NaN,NaN
3147,Martin Kerry,Male,M65-69,FIN,NaN,NaN,NaN,NaN
3148,Baris Kaya,Male,M45-49,FIN,NaN,NaN,NaN,NaN
3150,Javier Guerrero Gonzalez,Male,M30-34,FIN,NaN,NaN,NaN,NaN
3151,Olivier Lecerf,Male,M65-69,FIN,NaN,NaN,NaN,NaN
3152,Manfred Lambrechts,Male,M30-34,FIN,NaN,NaN,NaN,NaN
3153,Marcus Johnson,Male,M50-54,FIN,NaN,NaN,NaN,NaN
3154,Jeff Holt,Male,M70-74,FIN,NaN,NaN,NaN,NaN
3155,Fariz Miguel Kleber,Male,M40-44,FIN,NaN,NaN,NaN,NaN
3156,Sebastian Urra,Male,M18-24,FIN,NaN,NaN,NaN,NaN


Approximately 336 entries contained incomplete timing information. Some were labeled as finishers despite having no recorded result on the official website. These observations were excluded rather than imputed because their performance profiles could not be reliably reconstructed.

In [5]:
finishers = df[
    df["Status"] == "FIN"
].dropna(
    subset=["Swim Time", "Bike Time", "Run Time"]
).copy()
finishers = finishers[
    ["Athlete Name"] +
    [column for column in finishers.columns if "Seconds" in column]
]

time_columns = [
    column for column in finishers.columns
    if "Seconds" in column
]

times = finishers[time_columns].copy()
times = times[["Swim (Seconds)", "Bike (Seconds)", "Run (Seconds)"]]
times

,Swim (Seconds),Bike (Seconds),Run (Seconds)
0,1424.0,8210.0,4041.0
1,1430.0,8342.0,3912.0
2,1419.0,8331.0,4032.0
3,1480.0,8203.0,4131.0
4,1421.0,8233.0,4222.0
...,...,...,...
3374,2917.0,15894.0,12654.0
3397,2860.0,15478.0,11072.0
3464,3536.0,14363.0,8032.0
3465,2584.0,16187.0,12323.0


In [6]:
times.dtypes

Swim (Seconds)    float64
Bike (Seconds)    float64
Run (Seconds)     float64
dtype: object

In [7]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaled_times = scaler.fit_transform(times)

In [8]:
scaled_times = pd.DataFrame(
    scaled_times,
    columns=times.columns,
    index=times.index
)
scaled_times

,Swim (Seconds),Bike (Seconds),Run (Seconds)
0,-2.126615,-2.298139,-1.923614
1,-2.109522,-2.205734,-2.035230
2,-2.140860,-2.213434,-1.931401
3,-1.967075,-2.303040,-1.845742
4,-2.135162,-2.282038,-1.767005
...,...,...,...
3374,2.126838,3.081005,5.528729
3397,1.964449,2.789786,4.159914
3464,3.890326,2.009236,1.529574
3465,1.178144,3.286118,5.242334


removing strength

In [9]:
athlete_average = scaled_times.mean(axis = 1)

In [10]:
strengths = scaled_times.sub(athlete_average, axis=0) * -1
strengths

,Swim (Seconds),Bike (Seconds),Run (Seconds)
0,0.010492,0.182017,-0.192509
1,-0.007307,0.088905,-0.081598
2,0.045628,0.118202,-0.163831
3,-0.071544,0.264421,-0.192877
4,0.073760,0.220637,-0.294397
...,...,...,...
3374,1.452020,0.497853,-1.949872
3397,1.006934,0.181597,-1.188531
3464,-1.413947,0.467142,0.946805
3465,2.057388,-0.050586,-2.006802


In [11]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [12]:
k_values = range(2, 9)
inertias = []
silhouette_scores = []

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(strengths)

    inertias.append(model.inertia_)
    silhouette_scores.append(
        silhouette_score(strengths, labels)
    )

results = pd.DataFrame({
    "K": list(k_values),
    "Inertia": inertias,
    "Silhouette Score": silhouette_scores
})

results

,K,Inertia,Silhouette Score
0,2,1395.483296,0.348687
1,3,1097.988864,0.318884
2,4,860.351130,0.330987
3,5,731.262589,0.304297
4,6,641.032677,0.303232
5,7,573.045711,0.312406
6,8,508.827579,0.315459


In [13]:
kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=20
)

finishers["Cluster"] = kmeans.fit_predict(strengths)

In [14]:
centers = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=strengths.columns
)

centers

,Swim (Seconds),Bike (Seconds),Run (Seconds)
0,-0.679731,0.203558,0.476174
1,0.468198,-0.538869,0.070671
2,0.802143,0.090539,-0.892682
3,-0.039447,0.101016,-0.061569


In [15]:
finishers

,Athlete Name,Finish (Seconds),Swim (Seconds),T1 (Seconds),Bike (Seconds),T2 (Seconds),Run (Seconds),Cluster
0,Hayden Wilde,13851.0,1424.0,135.0,8210.0,41.0,4041.0,3
1,Kristian Blummenfelt,13869.0,1430.0,135.0,8342.0,51.0,3912.0,3
2,Jelle Geens,13960.0,1419.0,123.0,8331.0,55.0,4032.0,3
3,Gustav Iden,13996.0,1480.0,128.0,8203.0,53.0,4131.0,3
4,Lasse Nygaard Priester,14048.0,1421.0,120.0,8233.0,51.0,4222.0,3
...,...,...,...,...,...,...,...,...
3374,Giovanni Pastorelli,32598.0,2917.0,605.0,15894.0,529.0,12654.0,2
3397,Andrew Cox,30687.0,2860.0,724.0,15478.0,553.0,11072.0,2
3464,Khalid Oshman,27043.0,3536.0,675.0,14363.0,437.0,8032.0,0
3465,Paul Thelwell,32147.0,2584.0,442.0,16187.0,611.0,12323.0,2


In [16]:
cluster_names = {
    0: "Run-oriented",
    1: "Swim-oriented / Bike-weak",
    2: "Front-loaded / Run-weak",
    3: "Balanced"
}

In [17]:
time_columns = [
    "Swim (Seconds)",
    "Bike (Seconds)",
    "Run (Seconds)"
]

athletes = df.loc[
    (df["Status"] == "FIN") &
    df[time_columns].notna().all(axis=1),
    ["Athlete Name"] + time_columns
].copy()
athletes[
    ["Swim Strength", "Bike Strength", "Run Strength"]
] = strengths.to_numpy()

athletes["Cluster"] = kmeans.labels_
athletes["Archetype"] = athletes["Cluster"].map(cluster_names)

In [18]:
import plotly.express as px

In [19]:
import plotly.express as px

fig = px.scatter_3d(
    athletes,
    x="Swim Strength",
    y="Bike Strength",
    z="Run Strength",
    color="Archetype",
    hover_name="Athlete Name",
    title="Triathlete Performance Archetypes — Nice 2026"
)

fig.update_traces(marker=dict(size=4, opacity=0.7))
fig.show()

In [20]:
def find_athlete(name):
    return athletes.loc[
        athletes["Athlete Name"].str.contains(
            name,
            case=False,
            na=False
        ),
        [
            "Athlete Name",
            "Archetype",
            "Cluster",
            "Swim Strength",
            "Bike Strength",
            "Run Strength"
        ]
    ]
find_athlete("")

,Athlete Name,Archetype,Cluster,Swim Strength,Bike Strength,Run Strength
0,Hayden Wilde,Balanced,3,0.010492,0.182017,-0.192509
1,Kristian Blummenfelt,Balanced,3,-0.007307,0.088905,-0.081598
2,Jelle Geens,Balanced,3,0.045628,0.118202,-0.163831
3,Gustav Iden,Balanced,3,-0.071544,0.264421,-0.192877
4,Lasse Nygaard Priester,Balanced,3,0.073760,0.220637,-0.294397
...,...,...,...,...,...,...
3374,Giovanni Pastorelli,Front-loaded / Run-weak,2,1.452020,0.497853,-1.949872
3397,Andrew Cox,Front-loaded / Run-weak,2,1.006934,0.181597,-1.188531
3464,Khalid Oshman,Run-oriented,0,-1.413947,0.467142,0.946805
3465,Paul Thelwell,Front-loaded / Run-weak,2,2.057388,-0.050586,-2.006802


In [21]:
finishers.head(70)

,Athlete Name,Finish (Seconds),Swim (Seconds),T1 (Seconds),Bike (Seconds),T2 (Seconds),Run (Seconds),Cluster
0,Hayden Wilde,13851.0,1424.0,135.0,8210.0,41.0,4041.0,3
1,Kristian Blummenfelt,13869.0,1430.0,135.0,8342.0,51.0,3912.0,3
2,Jelle Geens,13960.0,1419.0,123.0,8331.0,55.0,4032.0,3
3,Gustav Iden,13996.0,1480.0,128.0,8203.0,53.0,4131.0,3
4,Lasse Nygaard Priester,14048.0,1421.0,120.0,8233.0,51.0,4222.0,3
...,...,...,...,...,...,...,...,...
65,Marc-antoine Garguet,15946.0,1909.0,208.0,9022.0,189.0,4617.0,0
66,Felix Schiller,15961.0,1559.0,170.0,9380.0,163.0,4689.0,3
67,Matthew McGoey,15976.0,1483.0,154.0,9654.0,57.0,4627.0,1
68,Staf Vanhauwaert,15988.0,1579.0,284.0,9088.0,140.0,4897.0,3


In [22]:
import numpy as np

strength_columns = [
    "Swim Strength",
    "Bike Strength",
    "Run Strength"
]

strength_array = athletes[strength_columns].to_numpy()

profile_shares = np.exp(strength_array)
profile_shares = profile_shares / profile_shares.sum(axis=1, keepdims=True) * 100

athletes[
    ["Swim Profile", "Bike Profile", "Run Profile"]
] = profile_shares

In [23]:
import plotly.express as px

color_map = {
    "Run-oriented": "#EF476F",
    "Swim-oriented / Bike-weak": "#118AB2",
    "Front-loaded / Run-weak": "#7B2CBF",
    "Balanced": "#06D6A0"
}

fig = px.scatter_ternary(
    athletes,
    a="Swim Profile",
    b="Bike Profile",
    c="Run Profile",
    color="Archetype",
    hover_name="Athlete Name",
    hover_data={
        "Swim Profile": ":.1f",
        "Bike Profile": ":.1f",
        "Run Profile": ":.1f",
        "Cluster": True
    },
    color_discrete_map=color_map,
    title="Performance Archetypes at the 2026 IRONMAN 70.3 World Championship"
)

fig.update_traces(
    marker={
        "size": 6,
        "opacity": 0.55,
        "line": {"width": 0.3, "color": "white"}
    }
)

fig.update_layout(
    template="plotly_white",
    height=750,
    margin={"l": 50, "r": 50, "t": 100, "b": 50},
    legend_title_text="Athlete archetype",
    title={
        "x": 0.5,
        "xanchor": "center"
    }
)

fig.update_ternaries(
    aaxis_title="Swim orientation",
    baxis_title="Bike orientation",
    caxis_title="Run orientation"
)

fig.show()

In [24]:
fig.write_html(
    "triathlon_archetypes.html",
    include_plotlyjs="cdn",
    full_html=True
)

In [25]:
import plotly.graph_objects as go

def find_on_chart(name):
    selected = athletes[
        athletes["Athlete Name"].str.contains(
            name,
            case=False,
            na=False
        )
    ]

    if selected.empty:
        print("No athlete found.")
        return

    new_fig = go.Figure(fig)

    new_fig.add_trace(
        go.Scatterternary(
            a=selected["Swim Profile"],
            b=selected["Bike Profile"],
            c=selected["Run Profile"],
            mode="markers+text",
            text=selected["Athlete Name"],
            textposition="top center",
            marker={
                "size": 14,
                "color": "black",
                "symbol": "diamond",
                "line": {
                    "width": 2,
                    "color": "white"
                }
            },
            name="Selected athlete",
            hovertemplate=(
                "<b>%{text}</b><br>"
                "Swim profile: %{a:.1f}<br>"
                "Bike profile: %{b:.1f}<br>"
                "Run profile: %{c:.1f}"
                "<extra></extra>"
            )
        )
    )

    new_fig.show()

In [26]:
find_on_chart("Hayden")

In [27]:
valid_mask = (
    (df["Status"] == "FIN") &
    df[time_columns].notna().all(axis=1)
)

assert valid_mask.sum() == len(athletes)

athletes["Overall Rank"] = df.loc[
    valid_mask,
    "Overall Rank"
].to_numpy()

In [28]:
archetype_summary = athletes.groupby("Archetype").agg(
    Athletes=("Overall Rank", "count"),
    Median_Rank=("Overall Rank", "median"),
    Mean_Rank=("Overall Rank", "mean"),
    Q1_Rank=("Overall Rank", lambda x: x.quantile(0.25)),
    Q3_Rank=("Overall Rank", lambda x: x.quantile(0.75))
)

archetype_summary["Rank_IQR"] = (
    archetype_summary["Q3_Rank"] -
    archetype_summary["Q1_Rank"]
)

In [29]:
top_10_cutoff = athletes["Overall Rank"].quantile(0.10)

top_10_rates = (
    athletes.assign(
        Top_10=athletes["Overall Rank"] <= top_10_cutoff
    )
    .groupby("Archetype")["Top_10"]
    .mean()
    .mul(100)
)

archetype_summary["Top_10_Percent"] = top_10_rates

In [30]:
archetype_summary.sort_values("Median_Rank")

,Athletes,Median_Rank,Mean_Rank,Q1_Rank,Q3_Rank,Rank_IQR,Top_10_Percent
Archetype,,,,,,,
Balanced,1467,1146.0,1377.590320,522.50,1898.50,1376.00,16.564417
Run-oriented,740,1626.0,1857.447297,915.75,2287.00,1371.25,6.486486
Swim-oriented / Bike-weak,612,2233.5,2058.155229,1512.75,2719.25,1206.50,2.614379
Front-loaded / Run-weak,342,2387.5,5249.090643,1462.00,2942.75,1480.75,2.923977


In [31]:
successful_specialists = athletes[
    athletes["Archetype"] != "Balanced"
].sort_values("Overall Rank")

successful_specialists[
    [
        "Athlete Name",
        "Overall Rank",
        "Archetype",
        
        "Swim Strength",
        "Bike Strength",
        "Run Strength"
    ]
].head(20)

,Athlete Name,Overall Rank,Archetype,Swim Strength,Bike Strength,Run Strength
26,Jens Emil Sloth Nielsen,27,Run-oriented,-0.396992,0.252720,0.144273
54,Cenzino Lebot,55,Front-loaded / Run-weak,0.482118,0.280519,-0.762637
60,Martin Baeza,61,Swim-oriented / Bike-weak,0.389876,-0.196020,-0.193856
61,Levin Veale,62,Front-loaded / Run-weak,0.277658,0.356198,-0.633856
64,Benjamin Ueltschi,65,Run-oriented,-0.415138,0.272852,0.142286
65,Marc-antoine Garguet,66,Run-oriented,-0.555056,0.429763,0.125293
67,Matthew McGoey,68,Swim-oriented / Bike-weak,0.404400,-0.266853,-0.137547
78,Emmanuel Lejeune,79,Swim-oriented / Bike-weak,0.359445,-0.220204,-0.139242
79,Greg Van Avermaet,80,Run-oriented,-0.724466,0.713044,0.011423
81,Youhan Guillou,82,Swim-oriented / Bike-weak,0.450235,-0.220252,-0.229982


In [32]:
athletes["Overall Rank"].describe()

count     3161.000000
mean      2040.561848
std       6825.558583
min          1.000000
25%        791.000000
50%       1581.000000
75%       2371.000000
max      99999.000000
Name: Overall Rank, dtype: float64

In [33]:
athletes.nlargest(
    20,
    "Overall Rank"
)[
    ["Athlete Name", "Overall Rank", "Archetype"]
]

,Athlete Name,Overall Rank,Archetype
3149,Mike Nelson,99999,Front-loaded / Run-weak
3177,Alain Guérin,99999,Front-loaded / Run-weak
3189,Terry Gates,99999,Front-loaded / Run-weak
3217,Kazutoshi Miyokawa,99999,Front-loaded / Run-weak
3302,Gotthard Winkler,99999,Front-loaded / Run-weak
3312,Uwe Prein,99999,Balanced
3322,Michael Zinn,99999,Front-loaded / Run-weak
3340,Hans - Georg Wilkening,99999,Balanced
3352,Matthias Gensior,99999,Run-oriented
3371,Björn Maas,99999,Front-loaded / Run-weak


In [34]:
ranked_athletes = athletes[
    athletes["Overall Rank"].between(1, len(athletes))
].copy()

In [35]:
print("All complete athletes:", len(athletes))
print("Officially ranked athletes:", len(ranked_athletes))
print("Excluded unranked athletes:", len(athletes) - len(ranked_athletes))
print("Maximum valid rank:", ranked_athletes["Overall Rank"].max())

All complete athletes: 3161
Officially ranked athletes: 3146
Excluded unranked athletes: 15
Maximum valid rank: 3146


In [36]:
archetype_summary = ranked_athletes.groupby("Archetype").agg(
    Athletes=("Overall Rank", "count"),
    Median_Rank=("Overall Rank", "median"),
    Mean_Rank=("Overall Rank", "mean"),
    Q1_Rank=("Overall Rank", lambda x: x.quantile(0.25)),
    Q3_Rank=("Overall Rank", lambda x: x.quantile(0.75))
)

archetype_summary["Rank_IQR"] = (
    archetype_summary["Q3_Rank"] -
    archetype_summary["Q1_Rank"]
)

In [37]:
top_10_cutoff = ranked_athletes["Overall Rank"].quantile(0.10)

top_10_rates = (
    ranked_athletes.assign(
        Top_10=ranked_athletes["Overall Rank"] <= top_10_cutoff
    )
    .groupby("Archetype")["Top_10"]
    .mean()
    .mul(100)
)

archetype_summary["Top_10_Percent"] = top_10_rates

archetype_summary.sort_values("Median_Rank")

,Athletes,Median_Rank,Mean_Rank,Q1_Rank,Q3_Rank,Rank_IQR,Top_10_Percent
Archetype,,,,,,,
Balanced,1465,1141.0,1242.953584,522.00,1895.00,1373.0,16.450512
Run-oriented,738,1624.5,1591.481030,915.25,2283.75,1368.5,6.504065
Swim-oriented / Bike-weak,612,2233.5,2058.155229,1512.75,2719.25,1206.5,2.614379
Front-loaded / Run-weak,331,2321.0,2100.302115,1457.50,2882.00,1424.5,3.021148
